## Parquet plus a transaction log

Plain data lakes store files — Parquet, JSON, CSV — in object storage like S3 or ADLS. Cheap and scalable, but no atomicity, no schema enforcement, no undo. A failed write leaves partial data behind. Two concurrent writes corrupt each other.

**Delta Lake** is an open-source storage layer that adds a **transaction log** on top of Parquet files. That log is what makes a Delta table different from a folder of Parquet files:

```text
Plain Parquet lake                   Delta Lake
──────────────────────               ─────────────────────────────────
s3://bucket/transactions/            s3://bucket/transactions/
  part-0001.parquet                    _delta_log/
  part-0002.parquet                      00000000000000000000.json  ← txn 0
  part-CORRUPT.parquet  ← ???            00000000000000000001.json  ← txn 1
  (no history, no undo)                  00000000000000000002.json  ← txn 2
                                       part-0001.parquet
                                       part-0002.parquet
                                         (every file accounted for in the log)
```

This notebook builds Delta from the ground up: log mechanics, ACID writes, schema enforcement and evolution, mutations (`DELETE` / `UPDATE` / `MERGE`), time travel, `VACUUM`, change data feed, and the optimization toolkit (`OPTIMIZE`, `ZORDER`, bloom filters, partitioning, generated columns, constraints, `CLONE`).


## Setup

One SparkSession with the Delta extensions, one scratch directory, and the bank dataset loaded once. Every section below builds on these.


In [ ]:
import os, json, shutil, time
from datetime import datetime, timezone
from pyspark.sql import SparkSession
from pyspark.sql.types import (
    StructType, StructField, StringType, BooleanType, IntegerType,
    DoubleType, DecimalType, DateType, TimestampType,
)
from pyspark.sql.functions import (
    col, count, sum as _sum, avg, lit, current_timestamp,
    to_date, expr, when, concat, floor, rand,
)
from pyspark.sql.utils import AnalysisException
from delta import configure_spark_with_delta_pip
from delta.tables import DeltaTable

builder = (
    SparkSession.builder
    .appName("DeltaLake")
    .master("local[*]")
    .config("spark.sql.extensions",
            "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog",
            "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .config("spark.sql.shuffle.partitions", "4")
)
spark = configure_spark_with_delta_pip(builder).getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

DATA    = "data"
SCRATCH = os.path.abspath("_scratch_09")
shutil.rmtree(SCRATCH, ignore_errors=True)
os.makedirs(SCRATCH, exist_ok=True)

customers = spark.read.option("header", "true").schema(StructType([
    StructField("customer_id",   StringType(),    False),
    StructField("full_name",     StringType(),    False),
    StructField("email",         StringType(),    True),
    StructField("phone",         StringType(),    True),
    StructField("city",          StringType(),    True),
    StructField("state",         StringType(),    True),
    StructField("date_of_birth", DateType(),      True),
    StructField("kyc_verified",  BooleanType(),   False),
    StructField("created_at",    TimestampType(), False),
])).csv(f"{DATA}/customers")

card_txns = spark.read.schema(StructType([
    StructField("txn_id",      StringType(),      False),
    StructField("card_id",     StringType(),      False),
    StructField("customer_id", StringType(),      False),
    StructField("amount",      DecimalType(18,2), False),
    StructField("merchant",    StringType(),      True),
    StructField("category",    StringType(),      True),
    StructField("txn_ts",      TimestampType(),   False),
    StructField("status",      StringType(),      False),
    StructField("is_fraud",    BooleanType(),     False),
])).parquet(f"{DATA}/card_transactions")

loan_accounts = spark.read.option("multiLine", "true").schema(StructType([
    StructField("loan_id",       StringType(),      False),
    StructField("customer_id",   StringType(),      False),
    StructField("loan_type",     StringType(),      False),
    StructField("principal",     DecimalType(18,2), False),
    StructField("interest_rate", DoubleType(),      False),
    StructField("tenure_months", IntegerType(),     False),
    StructField("disbursed_on",  DateType(),        False),
    StructField("status",        StringType(),      False),
])).json(f"{DATA}/loan_accounts")

print(f"Spark {spark.version} ready")
print(f"customers : {customers.count():>5}, card_txns : {card_txns.count():>5}, loans : {loan_accounts.count():>5}")


## The transaction log

Every Delta table has a `_delta_log/` directory alongside its data files. This directory is the source of truth.

- A sequence of JSON files, one per committed transaction: `00000000000000000000.json`, `00000000000000000001.json`, …
- Periodically, a **checkpoint** Parquet file that compresses the full log state into one file (every 10 commits by default)
- Each entry records which data files are **added** and which are **removed** in that transaction

**A single log entry looks like:**
```json
{ "commitInfo": { "operation": "WRITE", "timestamp": 1704067200000, ... } }
{ "add": { "path": "part-00000-abc.parquet", "size": 204800, "stats": "{\"numRecords\": 1000}" } }
{ "add": { "path": "part-00001-def.parquet", "size": 198400, "stats": "{\"numRecords\": 987}"  } }
```

**Readers** load the latest checkpoint (if any), then replay subsequent JSON entries to compute the active file set = all `add`s minus all `remove`s. No directory listing needed — that's why Delta reads stay fast on huge tables.


## Creating and reading a Delta table

Writing Delta is identical to writing Parquet — change the format string. Spark creates the data files **and** the `_delta_log/` directory automatically.


In [ ]:
CUST_DELTA = os.path.join(SCRATCH, "customers_delta")

customers.write.format("delta").mode("overwrite").save(CUST_DELTA)

print("Directory contents:")
for entry in sorted(os.listdir(CUST_DELTA)):
    print(f"  {entry}")

log_dir = os.path.join(CUST_DELTA, "_delta_log")
print("\n_delta_log:")
for entry in sorted(os.listdir(log_dir)):
    size = os.path.getsize(os.path.join(log_dir, entry))
    print(f"  {entry}  ({size} bytes)")

# Peek inside transaction 0
print("\nEntries in 00000000000000000000.json:")
with open(os.path.join(log_dir, "00000000000000000000.json")) as f:
    for line in f:
        entry = json.loads(line)
        key = list(entry.keys())[0]
        if key == "commitInfo":
            print(f"  commitInfo  operation : {entry['commitInfo'].get('operation')}")
        elif key == "add":
            stats = json.loads(entry["add"].get("stats", "{}"))
            print(f"  add         {entry['add']['path'][-30:]}  rows={stats.get('numRecords','?')}")
        elif key == "protocol":
            p = entry["protocol"]
            print(f"  protocol    minReader={p['minReaderVersion']}  minWriter={p['minWriterVersion']}")
        elif key == "metaData":
            print(f"  metaData    schema declared")

# Read it back and register as a SQL table
spark.sql(f"CREATE TABLE IF NOT EXISTS customers_delta USING delta LOCATION '{CUST_DELTA}'")
spark.sql("SELECT city, count(*) AS n FROM customers_delta GROUP BY city ORDER BY n DESC LIMIT 5").show()


## Writing — append, overwrite, replaceWhere

Delta supports the same `mode` options as any Spark write, but with ACID guarantees:

| Mode | Behaviour | Atomicity |
|---|---|---|
| `append` | Add new rows; existing untouched | Yes |
| `overwrite` | Replace the entire table | Yes |
| `overwrite` + `replaceWhere` | Replace only rows matching a predicate | Yes — partition-level overwrite |
| `error` (default) | Fail if the table exists | — |
| `ignore` | No-op if the table exists | — |

Each write is a new transaction. If the write fails mid-way, none of the new files appear in the log and the table is unchanged.


In [ ]:
TXN_DELTA = os.path.join(SCRATCH, "card_txns_delta")

# Version 0 — initial write (first half of the data)
(card_txns.filter(col("txn_ts") < "2024-04-01")
    .write.format("delta")
    .partitionBy("category")
    .mode("overwrite")
    .save(TXN_DELTA))
print(f"v0: {spark.read.format('delta').load(TXN_DELTA).count()} rows")

# Version 1 — append the second half
(card_txns.filter(col("txn_ts") >= "2024-04-01")
    .write.format("delta")
    .partitionBy("category")
    .mode("append")
    .save(TXN_DELTA))
print(f"v1: {spark.read.format('delta').load(TXN_DELTA).count()} rows (after append)")

# Version 2 — replaceWhere: rewrite only TRAVEL+SUCCESS rows with status=VERIFIED
replacement = (card_txns
    .filter((col("category") == "TRAVEL") & (col("status") == "SUCCESS"))
    .withColumn("status", lit("VERIFIED")))

(replacement.write.format("delta")
    .partitionBy("category")
    .mode("overwrite")
    .option("replaceWhere", "category = 'TRAVEL'")
    .save(TXN_DELTA))

print("v2 — TRAVEL category status distribution:")
(spark.read.format("delta").load(TXN_DELTA)
    .filter(col("category") == "TRAVEL")
    .groupBy("status").count().orderBy("status").show())


## ACID guarantees in practice

- **Atomicity** — a write commits all its files or none. If the job crashes after writing 60 of 100 files, those 60 exist on disk but are never referenced in the log. Readers never see them.
- **Consistency** — schema enforcement (next section) keeps every committed row valid.
- **Isolation** — optimistic concurrency control: each writer reads the current version, makes changes, then attempts to commit. If another writer committed a conflicting change, the commit fails and retries.
- **Durability** — once the log JSON file is written to object storage, the transaction is permanent.

The atomicity demo below proves that orphan files (Parquet files written outside the log) are invisible to Delta — the log alone decides what's part of the table.


In [ ]:
ATOMIC = os.path.join(SCRATCH, "atomic_demo")

customers.limit(100).write.format("delta").mode("overwrite").save(ATOMIC)
before = spark.read.format("delta").load(ATOMIC).count()
print(f"Before orphan write : {before} rows")

# Drop a Parquet file directly into the table dir, bypassing the log
orphan = os.path.join(ATOMIC, "orphan_file.parquet")
customers.limit(50).write.mode("overwrite").parquet(orphan)

after = spark.read.format("delta").load(ATOMIC).count()
print(f"After orphan write  : {after} rows  ← unchanged, orphan invisible to Delta")
print(f"Orphan exists       : {os.path.exists(orphan)}")


## Schema enforcement

Delta **rejects writes that do not match the table schema** — at write time, not at read time.

- Extra columns not in the schema → rejected (unless `mergeSchema`)
- Missing non-nullable columns → rejected
- Wrong data type for an existing column → rejected


In [ ]:
ENFORCE = os.path.join(SCRATCH, "schema_enforce")
customers.write.format("delta").mode("overwrite").save(ENFORCE)

# Reject 1: extra column
extra = customers.withColumn("credit_score", lit(750))
try:
    extra.write.format("delta").mode("append").save(ENFORCE)
except AnalysisException as e:
    print("REJECTED — extra column:")
    print(f"  {str(e)[:140]}")

# Reject 2: wrong data type
wrong_type = customers.withColumn("kyc_verified", col("kyc_verified").cast("string"))
try:
    wrong_type.write.format("delta").mode("append").save(ENFORCE)
except Exception as e:
    print("\nREJECTED — wrong type:")
    print(f"  {str(e)[:140]}")

print(f"\nTable unchanged: {spark.read.format('delta').load(ENFORCE).count()} rows")


## Schema evolution — `mergeSchema`

Sometimes you **intentionally** want to add columns. Enable `mergeSchema = true`:

- New columns in the incoming data are added to the table schema
- Existing rows get `null` for the new column
- Existing columns are not removed
- Type changes are still rejected unless types are compatible

`mergeSchema` is opt-in per write — you must request it explicitly each time.


In [ ]:
EVOLVE = os.path.join(SCRATCH, "schema_evolve")
customers.write.format("delta").mode("overwrite").save(EVOLVE)
print("v0 columns:", spark.read.format("delta").load(EVOLVE).columns)

# Upstream now adds credit_score and risk_tier
customers_v2 = (customers
    .withColumn("credit_score",
                (700 + (col("customer_id").substr(5, 4).cast("int") % 200)).cast("int"))
    .withColumn("risk_tier", lit("STANDARD")))

(customers_v2.write.format("delta").mode("append")
    .option("mergeSchema", "true")
    .save(EVOLVE))

print("v1 columns:", spark.read.format("delta").load(EVOLVE).columns)

evolved = spark.read.format("delta").load(EVOLVE)
print("Old rows (credit_score is null):", evolved.filter(col("credit_score").isNull()).count())
print("New rows (credit_score set)    :", evolved.filter(col("credit_score").isNotNull()).count())


## DELETE FROM

`DELETE FROM` removes rows matching a predicate. Under the hood Delta does **not** modify Parquet files in place — it rewrites the affected files with the matching rows omitted and records the old files as `remove` entries in the log. The old files stay on disk until `VACUUM` removes them — which is what makes time travel possible.

We'll build a `loans` table here and reuse it through `UPDATE`, `MERGE`, time travel, and rollback.


In [ ]:
LOANS = os.path.join(SCRATCH, "loans")

# v0 — initial load
loan_accounts.write.format("delta").mode("overwrite").save(LOANS)
spark.sql(f"CREATE TABLE IF NOT EXISTS loans USING delta LOCATION '{LOANS}'")

loans_dt = DeltaTable.forPath(spark, LOANS)
print(f"v0: {spark.read.format('delta').load(LOANS).count()} rows")

# v1 — DELETE rejected loans
print("\nBefore DELETE:")
spark.sql("SELECT status, count(*) AS n FROM loans GROUP BY status ORDER BY n DESC").show()

loans_dt.delete(condition=col("status") == "REJECTED")
# SQL equivalent: spark.sql("DELETE FROM loans WHERE status = 'REJECTED'")

print("After DELETE (v1):")
spark.sql("SELECT status, count(*) AS n FROM loans GROUP BY status ORDER BY n DESC").show()


## UPDATE SET

`UPDATE` modifies column values for rows matching a predicate. Like `DELETE`, only files containing matching rows are rewritten; unaffected files are untouched. Multiple columns can be updated in a single statement, and the new value can be any expression.


In [ ]:
# v2 — close old ACTIVE loans and apply a 10% rate discount
loans_dt.update(
    condition=(col("status") == "ACTIVE") & (col("disbursed_on") < "2022-01-01"),
    set={
        "status":        lit("CLOSED"),
        "interest_rate": col("interest_rate") * 0.9,
    },
)

# v3 — SQL syntax: flag high-rate delinquents for review
spark.sql("""
    UPDATE loans
    SET    status = 'UNDER_REVIEW'
    WHERE  status = 'DELINQUENT' AND interest_rate > 12
""")

print("Status distribution after both UPDATEs:")
spark.sql("SELECT status, count(*) AS n FROM loans GROUP BY status ORDER BY n DESC").show()


## MERGE INTO — upsert in one transaction

`MERGE INTO` joins a **source** DataFrame against the **target** table on a match condition and applies different actions per row:

```sql
MERGE INTO target t
USING  source s
ON     t.id = s.id

WHEN MATCHED AND s.amount > 1000       -- matched + extra condition
  THEN UPDATE SET t.status = 'FLAGGED'

WHEN MATCHED                           -- matched (no extra condition)
  THEN UPDATE SET *

WHEN NOT MATCHED                       -- in source but not in target
  THEN INSERT *

WHEN NOT MATCHED BY SOURCE             -- in target but not in source
  THEN DELETE
```

All clauses are optional. Common patterns: SCD Type 1 upsert (`MATCHED → UPDATE`, `NOT MATCHED → INSERT`), conditional update (only overwrite if source is newer), full sync (add `NOT MATCHED BY SOURCE → DELETE` to mirror an upstream system).

The demo below covers all three actions in one MERGE.


In [ ]:
# Source: 2 existing loan_ids with updates + 2 brand new loans + the source omits 5 prior loan_ids
existing = spark.sql("SELECT loan_id FROM loans WHERE status = 'ACTIVE' LIMIT 2").collect()

incoming_schema = StructType([
    StructField("loan_id",       StringType(),      False),
    StructField("customer_id",   StringType(),      False),
    StructField("loan_type",     StringType(),      False),
    StructField("principal",     DecimalType(18,2), False),
    StructField("interest_rate", DoubleType(),      False),
    StructField("tenure_months", IntegerType(),     False),
    StructField("disbursed_on",  DateType(),        False),
    StructField("status",        StringType(),      False),
])

updates = [
    (r["loan_id"], "CUST0001", "PERSONAL", 50000.00, 11.5, 36,
     datetime(2024, 6, 1).date(), "ACTIVE")
    for r in existing
]
new_rows = [
    ("LN-NEW-001", "CUST0099", "HOME",    2500000.00, 8.75, 240,
     datetime(2024, 9, 15).date(), "ACTIVE"),
    ("LN-NEW-002", "CUST0100", "VEHICLE",  750000.00, 9.5,   60,
     datetime(2024, 10, 1).date(), "ACTIVE"),
]
source = spark.createDataFrame(updates + new_rows, schema=incoming_schema)

# Use a dedicated table so NOT MATCHED BY SOURCE works on a controlled subset
SYNC = os.path.join(SCRATCH, "loans_sync")
loan_accounts.limit(20).write.format("delta").mode("overwrite").save(SYNC)
sync_dt = DeltaTable.forPath(spark, SYNC)

# Source contains only 15 of the original 20 — the other 5 should be deleted
sync_source = loan_accounts.limit(15).unionByName(spark.createDataFrame(new_rows, schema=incoming_schema))

before = spark.read.format("delta").load(SYNC).count()
(sync_dt.alias("t")
    .merge(sync_source.alias("s"), "t.loan_id = s.loan_id")
    .whenMatchedUpdateAll()              # update existing
    .whenNotMatchedInsertAll()           # insert new
    .whenNotMatchedBySourceDelete()      # delete rows not in source
    .execute())
after = spark.read.format("delta").load(SYNC).count()
print(f"Sync table : before {before} rows  →  after {after} rows  (5 deleted, 2 inserted)")


## MERGE — SQL syntax

The same operations are available directly in Spark SQL. Useful when building pipelines as `spark.sql()` calls or writing transformations SQL-first.


In [ ]:
# Staging: some customers changed city/email, one is brand new
staging = spark.createDataFrame([
    ("CUST0001", "Alice Johnson", "alice.new@example.com", "555-0001",
     "New York", "NY", None, True, datetime(2024, 1, 1)),
    ("CUST9999", "New Customer", "new@example.com", "555-9999",
     "Austin", "TX", None, True, datetime(2024, 10, 1)),  # new
], schema=customers.schema)

staging.createOrReplaceTempView("customer_staging")

spark.sql("""
    MERGE INTO customers_delta AS t
    USING  customer_staging   AS s
    ON     t.customer_id = s.customer_id

    WHEN MATCHED THEN
      UPDATE SET t.full_name = s.full_name,
                 t.email     = s.email,
                 t.city      = s.city

    WHEN NOT MATCHED THEN
      INSERT (customer_id, full_name, email, phone, city, state,
              date_of_birth, kyc_verified, created_at)
      VALUES (s.customer_id, s.full_name, s.email, s.phone, s.city, s.state,
              s.date_of_birth, s.kyc_verified, s.created_at)
""")

spark.sql("""
    SELECT customer_id, full_name, email, city
    FROM   customers_delta
    WHERE  customer_id IN ('CUST0001', 'CUST9999')
""").show(truncate=False)


## Time travel and table history

Delta retains old files until `VACUUM` removes them, so you can read any past version of a table — by version number or by timestamp.

| Method | DataFrame API | SQL |
|---|---|---|
| By version number | `.option("versionAsOf", N)` | `VERSION AS OF N` |
| By timestamp | `.option("timestampAsOf", "2024-01-01")` | `TIMESTAMP AS OF '2024-01-01'` |

`DESCRIBE HISTORY` (SQL) or `DeltaTable.history()` (Python) returns a full audit trail — operation, timestamp, parameters, and metrics — useful for finding the version to time-travel to.


In [ ]:
# Full history of the loans table
loans_dt.history().select(
    "version", "timestamp", "operation",
    col("operationMetrics.numOutputRows").alias("rows"),
).show(truncate=False)

# Time travel — DataFrame API
v0 = spark.read.format("delta").option("versionAsOf", 0).load(LOANS)
v1 = spark.read.format("delta").option("versionAsOf", 1).load(LOANS)
v_now = spark.read.format("delta").load(LOANS)

print(f"v0  (initial)         : {v0.count()} rows")
print(f"v1  (after DELETE)    : {v1.count()} rows")
print(f"now (after all UPDATEs): {v_now.count()} rows")
print(f"REJECTED in v0 : {v0.filter(col('status') == 'REJECTED').count()}")
print(f"REJECTED in v1 : {v1.filter(col('status') == 'REJECTED').count()} ← deleted")

# Time travel — SQL syntax
print("\nv0 status distribution (via SQL VERSION AS OF):")
spark.sql("SELECT status, count(*) AS n FROM loans VERSION AS OF 0 GROUP BY status ORDER BY n DESC").show()


## Rollback and `RESTORE TABLE`

There's no single-command rollback in open-source Delta, but the pattern is simple — **time-travel read the past version, then overwrite the current table with it**. The rollback is itself a new transaction, so the history of both the mistake and the recovery is preserved.

On Databricks (and Delta 1.2+) you can use the single-command form:

```sql
RESTORE TABLE loans TO VERSION AS OF 0
RESTORE TABLE loans TO TIMESTAMP AS OF '2024-01-01 09:00:00'
```


In [ ]:
ROLLBACK = os.path.join(SCRATCH, "loans_rollback")
loan_accounts.write.format("delta").mode("overwrite").save(ROLLBACK)
rb_dt = DeltaTable.forPath(spark, ROLLBACK)
print(f"v0 count: {spark.read.format('delta').load(ROLLBACK).count()}")

# v1: bad delete — accidentally removed all PERSONAL loans
rb_dt.delete(col("loan_type") == "PERSONAL")
print(f"v1 count (bad delete): {spark.read.format('delta').load(ROLLBACK).count()}")

# Rollback: read v0, overwrite current
v0_snapshot = spark.read.format("delta").option("versionAsOf", 0).load(ROLLBACK)
(v0_snapshot.write.format("delta").mode("overwrite")
    .option("overwriteSchema", "true").save(ROLLBACK))

print(f"v2 count (after rollback): {spark.read.format('delta').load(ROLLBACK).count()}")

# Both the mistake and the recovery are visible in history
DeltaTable.forPath(spark, ROLLBACK).history().select(
    "version", "operation"
).show()


## VACUUM and retention

Delta marks old files as `remove` in the log but does not delete them from disk — those files are what enable time travel. `VACUUM` physically deletes files that are:

1. Marked as removed in the log, **and**
2. Older than the **retention threshold** (default: 7 days)

```sql
VACUUM loans RETAIN 168 HOURS   -- keep 7 days (default)
VACUUM loans RETAIN 0 HOURS     -- delete everything beyond current version
```

After `VACUUM`, time travel to versions whose files were deleted becomes impossible — but the log entries still exist. Delta blocks `VACUUM` with retention below 7 days unless you disable the safety check (`spark.databricks.delta.retentionDurationCheck.enabled=false`) — testing only, never production.

Two table properties control retention:

| Property | Default | Controls |
|---|---|---|
| `delta.logRetentionDuration` | 30 days | How long log entries are kept |
| `delta.deletedFileRetentionDuration` | 7 days | How long removed data files are kept on disk |


In [ ]:
def count_parquet(path):
    n = 0
    for root, _, files in os.walk(path):
        if "_delta_log" in root: continue
        n += sum(1 for f in files if f.endswith(".parquet"))
    return n

print(f"Parquet files before VACUUM: {count_parquet(LOANS)}")

# DRY RUN — list what would be deleted without deleting
spark.conf.set("spark.databricks.delta.retentionDurationCheck.enabled", "false")
dry = spark.sql("VACUUM loans RETAIN 0 HOURS DRY RUN")
print(f"VACUUM would remove: {dry.count()} files")

# Execute VACUUM
spark.sql("VACUUM loans RETAIN 0 HOURS")
print(f"Parquet files after VACUUM:  {count_parquet(LOANS)}")

# Time travel to v0 now fails — files are gone
try:
    spark.read.format("delta").option("versionAsOf", 0).load(LOANS).count()
except Exception as e:
    print(f"\nv0 read fails (expected): {str(e)[:120]}")
print(f"Current version still readable: {spark.read.format('delta').load(LOANS).count()} rows")

spark.conf.set("spark.databricks.delta.retentionDurationCheck.enabled", "true")

# Set retention properties on the loans table
spark.sql("""
    ALTER TABLE loans SET TBLPROPERTIES (
        'delta.logRetentionDuration'         = 'interval 90 days',
        'delta.deletedFileRetentionDuration' = 'interval 30 days'
    )
""")
spark.sql("SHOW TBLPROPERTIES loans").filter(col("key").startswith("delta.")).show(truncate=False)


## Change Data Feed

**Change Data Feed (CDF)** tracks row-level changes to a Delta table — which rows were inserted, updated (with before and after images), or deleted. CDF is the foundation for CDC pipelines: downstream consumers read only what changed instead of re-reading the entire table.

Enable per table:

```sql
ALTER TABLE loans SET TBLPROPERTIES ('delta.enableChangeDataFeed' = 'true')
```

CDF adds three system columns:

| Column | Values | Meaning |
|---|---|---|
| `_change_type` | `insert` / `update_preimage` / `update_postimage` / `delete` | What happened |
| `_commit_version` | int | Which version produced the change |
| `_commit_timestamp` | timestamp | When the commit happened |

CDF works as both a batch read and a streaming source.


In [ ]:
CDF = os.path.join(SCRATCH, "loans_cdf")

(loan_accounts.write.format("delta").mode("overwrite")
    .option("delta.enableChangeDataFeed", "true")
    .save(CDF))
spark.sql(f"CREATE TABLE IF NOT EXISTS loans_cdf USING delta LOCATION '{CDF}'")
cdf_dt = DeltaTable.forPath(spark, CDF)

# v1 — delete REJECTED
cdf_dt.delete(col("status") == "REJECTED")

# v2 — flag high-rate ACTIVE loans
spark.sql("""
    UPDATE loans_cdf SET status = 'HIGH_RISK'
    WHERE  status = 'ACTIVE' AND interest_rate > 14
""")

# v3 — append two new loans
new_loans = spark.createDataFrame([
    ("LN-CDF-001", "CUST0001", "PERSONAL", 100000.00, 10.5, 24,
     datetime(2024, 11, 1).date(), "ACTIVE"),
    ("LN-CDF-002", "CUST0002", "VEHICLE",   50000.00,  9.8, 48,
     datetime(2024, 11, 1).date(), "ACTIVE"),
], schema=loan_accounts.schema)
new_loans.write.format("delta").mode("append").save(CDF)

# Read all changes since v1
changes = (spark.read.format("delta")
    .option("readChangeFeed",  "true")
    .option("startingVersion", 1)
    .load(CDF))

changes.groupBy("_change_type", "_commit_version").count() \
    .orderBy("_commit_version", "_change_type").show()


## Optimistic concurrency

Delta uses **optimistic concurrency control**: multiple writers run simultaneously without blocking each other. When a writer commits, Delta checks whether any conflicting change was committed since the writer started. If yes, the commit fails and the writer can retry.

| Operation | Conflicts with |
|---|---|
| Append | Rarely conflicts — appends are almost always compatible |
| Update / Delete | Conflicts if another writer touched the same files |
| MERGE | Conflicts if another writer modified files MERGE needs to read or write |
| Schema change | Conflicts with any concurrent writer |

**Partition-level isolation:** if two writers update *different* partitions, Delta detects this and lets both commit without conflict.

```
Writer A: UPDATE loans WHERE loan_type = 'HOME'    → touches HOME partition
Writer B: UPDATE loans WHERE loan_type = 'VEHICLE' → touches VEHICLE partition
No overlap → both succeed.

Writer C: UPDATE loans WHERE status = 'ACTIVE'     → touches ALL partitions
Writer D: UPDATE loans WHERE loan_type = 'HOME'    → starts while C runs
C commits first → D's commit fails on the HOME files → D retries against the new version → succeeds.
```


## The small-file problem and `OPTIMIZE`

Every Spark write produces files — one per output partition per task. Streaming pipelines make this worse. Over time a table can accumulate thousands of tiny files.

```text
1 000 files × 1 MB each = 1 GB total
Opening a file in S3 costs ~10 ms (LIST + GET + open).
1 000 × 10 ms = 10 s of overhead before reading a single row.
Same data in 10 files × 100 MB → 0.1 s of overhead.
```

`OPTIMIZE` reads small files in a partition, combines their rows, and writes fewer larger files (target: 1 GB each). The old files are marked `remove` and the new compacted files are `add`ed in one atomic transaction.

```sql
OPTIMIZE table_name                               -- compact all partitions
OPTIMIZE table_name WHERE date = '2024-01-01'     -- one partition only
OPTIMIZE table_name ZORDER BY (col1, col2)        -- compact + cluster
```


In [ ]:
SMALL = os.path.join(SCRATCH, "txns_small")

# Force 32 small files
(card_txns.repartition(32).write.format("delta").mode("overwrite").save(SMALL))
spark.sql(f"CREATE TABLE IF NOT EXISTS txns_small USING delta LOCATION '{SMALL}'")

n_before = count_parquet(SMALL)
print(f"Before OPTIMIZE: {n_before} files")

result = spark.sql("OPTIMIZE txns_small")
result.select(
    col("metrics.numFilesAdded").alias("files_added"),
    col("metrics.numFilesRemoved").alias("files_removed"),
).show()

print(f"After OPTIMIZE : {count_parquet(SMALL)} active files (old marked remove in log)")


## Data skipping

When Delta writes a Parquet file it records column-level statistics in the transaction log:

- `minValues`, `maxValues` — per column, per file
- `nullCount` — per column
- `numRecords` — total rows

At query time, if a filter predicate is provably false for a file based on its stats, Spark skips the file without opening it.

```
Query: WHERE amount > 500
  File A: min=10, max=490   → max < 500 → SKIP
  File B: min=50, max=650   → max > 500 → READ
  File C: min=10, max=300   → SKIP
  File D: min=200, max=800  → READ
```

Skipping is most effective when data is physically clustered by the filter column — which is exactly what `ZORDER BY` provides.


In [ ]:
# Peek at per-file stats in the transaction log
log_dir = os.path.join(SMALL, "_delta_log")
log_file = os.path.join(log_dir, "00000000000000000000.json")

print("Per-file stats from the log (first 3 files):\n")
shown = 0
with open(log_file) as f:
    for line in f:
        entry = json.loads(line)
        if "add" in entry and shown < 3:
            stats = json.loads(entry["add"].get("stats", "{}"))
            mv = stats.get("minValues", {})
            xv = stats.get("maxValues", {})
            print(f"File: {entry['add']['path'][-30:]}")
            print(f"  numRecords : {stats.get('numRecords')}")
            for c in ["amount", "txn_ts"]:
                if c in mv:
                    print(f"  {c}: min={mv[c]}  max={xv[c]}")
            print()
            shown += 1


## ZORDER BY — multi-dimensional clustering

`ZORDER BY` co-locates related data in the same files using a space-filling curve. After `OPTIMIZE … ZORDER BY (col1, col2)`, files contain narrow ranges of `col1` **and** `col2` simultaneously — so data skipping benefits both filters.

A regular sort on `(category, amount)` clusters well for `category` but not for `amount` alone. Z-ordering interleaves the keys so both dimensions skip well.

**Rules for choosing ZORDER columns:**
- Columns that appear in `WHERE` clauses
- High cardinality (`customer_id`, `txn_id`) — Z-ordering on a boolean is useless
- Two to four columns max — more dilutes the benefit
- Do **not** Z-ORDER by partition columns (already handled by directory pruning)
- Z-ORDER is re-run every `OPTIMIZE` — not maintained incrementally


In [ ]:
ZO = os.path.join(SCRATCH, "txns_zorder")
NO = os.path.join(SCRATCH, "txns_noorder")

for path in (ZO, NO):
    card_txns.repartition(16).write.format("delta").mode("overwrite").save(path)
spark.sql(f"CREATE TABLE IF NOT EXISTS txns_zorder USING delta LOCATION '{ZO}'")
spark.sql(f"CREATE TABLE IF NOT EXISTS txns_noorder USING delta LOCATION '{NO}'")

spark.sql("OPTIMIZE txns_noorder")
spark.sql("OPTIMIZE txns_zorder ZORDER BY (customer_id, amount)")

target = "CUST0005"
pred = (col("customer_id") == target) & (col("amount") > 400)

# Warm both
spark.read.format("delta").load(NO).filter(pred).count()
spark.read.format("delta").load(ZO).filter(pred).count()

t0 = time.time(); n_no = spark.read.format("delta").load(NO).filter(pred).count(); t_no = time.time() - t0
t0 = time.time(); n_zo = spark.read.format("delta").load(ZO).filter(pred).count(); t_zo = time.time() - t0

print(f"Filter: customer_id = '{target}' AND amount > 400")
print(f"  OPTIMIZE only    : {n_no} rows in {t_no:.2f}s")
print(f"  OPTIMIZE+ZORDER  : {n_zo} rows in {t_zo:.2f}s  ← more files skipped")


## Bloom filter indexes

A **bloom filter** is a probabilistic structure that answers "is this value in this file?" with two answers:

- **Definitely not** — skip the file
- **Possibly yes** — read the file (false positives possible, no false negatives)

Stored per file in `_delta_log/`, a few KB even for millions of distinct values.

**When to use:** equality lookups on high-cardinality columns (UUID, txn_id) — Z-ORDER's min/max stats are weak here because a UUID has a wide min/max range but only one file actually contains any given value. **Don't use** for low-cardinality columns or range queries.

```sql
CREATE BLOOMFILTER INDEX ON TABLE txns
FOR COLUMNS (txn_id OPTIONS (fpp=0.01, numItems=1000000))
```

`fpp` = false positive probability. `numItems` = expected distinct values (sizes the filter).


In [ ]:
BLOOM = os.path.join(SCRATCH, "txns_bloom")
card_txns.write.format("delta").mode("overwrite").save(BLOOM)
spark.sql(f"CREATE TABLE IF NOT EXISTS txns_bloom USING delta LOCATION '{BLOOM}'")

spark.sql("""
    CREATE BLOOMFILTER INDEX ON TABLE txns_bloom
    FOR COLUMNS (txn_id OPTIONS (fpp=0.01, numItems=50000))
""")
spark.sql("OPTIMIZE txns_bloom ZORDER BY (txn_id)")

# Point lookup
sample_id = card_txns.select("txn_id").limit(1).collect()[0][0]
print(f"Looking up txn_id = '{sample_id}'")
n = spark.sql(f"SELECT count(*) FROM txns_bloom WHERE txn_id = '{sample_id}'").collect()[0][0]
print(f"Bloom-filtered lookup found {n} row(s) — most files skipped without reading")


## Partition strategies

Partitioning physically separates data into subdirectories. A bad partition choice hurts more than not partitioning at all.

| Rule | Why |
|---|---|
| Low-to-medium cardinality (10–10 000) | Too few = no benefit. Too many = millions of tiny dirs |
| Frequently filtered | Queries must use the partition column in `WHERE` for pruning to work |
| Balanced distribution | Skewed partitions cause slow tasks |
| Target ≥1 GB per partition | Less = small-file problem |
| Never partition by high-cardinality columns | `customer_id` (10 000 unique) → 10 000 directories of ~50 rows each |

Most common good partition columns: `date`, `month`, `region`, `status`, `category`.


## Generated columns

A **generated column** is computed automatically from other columns by a deterministic expression. The most common use: derive a `date` partition key from a timestamp column so writers only need to provide the raw timestamp.

```sql
CREATE TABLE card_txns (
  txn_id   STRING NOT NULL,
  txn_ts   TIMESTAMP NOT NULL,
  txn_date DATE GENERATED ALWAYS AS (CAST(txn_ts AS DATE)),
  amount   DECIMAL(18,2),
  ...
) USING delta PARTITIONED BY (txn_date)
```

When you write a DataFrame with `txn_ts` but without `txn_date`, Delta computes `txn_date` automatically and partition pruning works on it.


In [ ]:
GEN = os.path.join(SCRATCH, "txns_gen_col")

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS txns_gen_col (
        txn_id      STRING NOT NULL,
        card_id     STRING NOT NULL,
        customer_id STRING NOT NULL,
        amount      DECIMAL(18,2) NOT NULL,
        merchant    STRING,
        category    STRING,
        txn_ts      TIMESTAMP NOT NULL,
        status      STRING,
        is_fraud    BOOLEAN,
        txn_date    DATE GENERATED ALWAYS AS (CAST(txn_ts AS DATE))
    )
    USING delta PARTITIONED BY (txn_date)
    LOCATION '{GEN}'
""")

# Write WITHOUT txn_date — Delta derives it
card_txns.write.format("delta").mode("append").saveAsTable("txns_gen_col")

spark.sql("SELECT txn_id, txn_ts, txn_date FROM txns_gen_col LIMIT 5").show()
spark.sql("""
    SELECT count(*) AS n, round(sum(amount), 2) AS total
    FROM   txns_gen_col
    WHERE  txn_date = '2024-03-15'
""").show()


## Constraints — NOT NULL and CHECK

Delta supports two value-level constraints, both enforced at write time:

| Constraint | SQL | Description |
|---|---|---|
| `NOT NULL` | Declared in `CREATE TABLE` column definition | Column cannot be null |
| `CHECK` | `ALTER TABLE ADD CONSTRAINT name CHECK (expr)` | Arbitrary boolean expression must hold |

Constraints are stored as table properties and enforced by every writer — not just one job. They act as a schema-level data quality gate that catches bad data at the source.


In [ ]:
CONST = os.path.join(SCRATCH, "loans_constrained")

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS loans_constrained (
        loan_id       STRING NOT NULL,
        customer_id   STRING NOT NULL,
        loan_type     STRING NOT NULL,
        principal     DECIMAL(18,2) NOT NULL,
        interest_rate DOUBLE NOT NULL,
        tenure_months INT NOT NULL,
        disbursed_on  DATE NOT NULL,
        status        STRING NOT NULL
    )
    USING delta LOCATION '{CONST}'
""")

spark.sql("ALTER TABLE loans_constrained ADD CONSTRAINT principal_positive CHECK (principal > 0)")
spark.sql("ALTER TABLE loans_constrained ADD CONSTRAINT interest_range CHECK (interest_rate BETWEEN 1.0 AND 30.0)")
spark.sql("ALTER TABLE loans_constrained ADD CONSTRAINT valid_status CHECK (status IN ('ACTIVE','CLOSED','DELINQUENT','UNDER_REVIEW'))")

# Good write — passes all constraints
loan_accounts.write.format("delta").mode("append").saveAsTable("loans_constrained")
print(f"Good write succeeded: {spark.sql('SELECT count(*) FROM loans_constrained').collect()[0][0]} rows")

# Bad write — invalid status
bad = spark.createDataFrame([
    ("LN-BAD", "CUST0001", "PERSONAL", 5000.00, 10.0, 24,
     datetime(2024, 1, 1).date(), "PENDING")
], schema=loan_accounts.schema)
try:
    bad.write.format("delta").mode("append").saveAsTable("loans_constrained")
except Exception as e:
    print(f"Bad write REJECTED (invalid status): {str(e)[:120]}")


## ANALYZE TABLE

`ANALYZE TABLE` computes column-level statistics (distinct count, min, max, null count) and stores them in the metastore. The Spark optimizer (Catalyst) uses these to pick the right join strategy and estimate result sizes — without them it falls back to rough estimates and may pick a sort-merge join when a broadcast would be faster.

```sql
ANALYZE TABLE loans COMPUTE STATISTICS                        -- row count + size
ANALYZE TABLE loans COMPUTE STATISTICS FOR ALL COLUMNS         -- full column stats
ANALYZE TABLE loans COMPUTE STATISTICS FOR COLUMNS status, type -- specific
```


In [ ]:
spark.sql("ANALYZE TABLE loans_constrained COMPUTE STATISTICS")
spark.sql("""
    ANALYZE TABLE loans_constrained
    COMPUTE STATISTICS FOR COLUMNS status, loan_type, principal, interest_rate
""")

spark.sql("DESCRIBE EXTENDED loans_constrained status").show(truncate=False)


## Auto-compaction and optimized writes (Databricks)

On Databricks, two table-level features manage file sizes automatically:

- **Auto-compaction** — after each write, if enough small files have accumulated in a partition, Delta runs a compaction in the background. The table stays in a reasonably compacted state without explicit `OPTIMIZE` calls.
- **Optimized writes** — Delta coalesces output files at write time to target ~1 GB each, instead of one file per Spark task. Prevents the small-file problem at the source.

```sql
ALTER TABLE loans SET TBLPROPERTIES (
  'delta.autoOptimize.optimizeWrite' = 'true',
  'delta.autoOptimize.autoCompact'   = 'true'
)
```

Both are Databricks-only. On open-source Spark, schedule `OPTIMIZE` periodically instead.


## CLONE — shallow and deep copies

`CLONE` creates a copy of a Delta table.

| Mode | Cost | Data files copied | Use for |
|---|---|---|---|
| **Shallow** | Cheap (metadata only) | No — references source files | Dev/test, short-lived branches |
| **Deep** | Expensive (copies all files) | Yes — fully independent copy | DR, archiving, migrating |

**Shallow clone** behaves like a normal Delta table: it has its own log, you can run `DELETE`, `UPDATE`, `MERGE` on it, and Delta copies files on write (copy-on-write semantics) — the source remains untouched.

```sql
CREATE TABLE loans_clone  SHALLOW CLONE loans AT VERSION AS OF 3
CREATE TABLE loans_backup DEEP    CLONE loans
```


In [ ]:
SHALLOW = os.path.join(SCRATCH, "loans_shallow")
DEEP    = os.path.join(SCRATCH, "loans_deep")

t0 = time.time()
spark.sql(f"CREATE OR REPLACE TABLE loans_shallow SHALLOW CLONE loans_constrained LOCATION '{SHALLOW}'")
t_shallow = time.time() - t0

t0 = time.time()
spark.sql(f"CREATE OR REPLACE TABLE loans_deep DEEP CLONE loans_constrained LOCATION '{DEEP}'")
t_deep = time.time() - t0

print(f"Shallow clone : {t_shallow:.2f}s  (metadata only)")
print(f"Deep    clone : {t_deep:.2f}s  (all files copied)")

# Mutate the shallow clone — source is unaffected
src_before = spark.sql("SELECT count(*) FROM loans_constrained").collect()[0][0]
spark.sql("DELETE FROM loans_shallow WHERE status = 'CLOSED'")
clone_after = spark.sql("SELECT count(*) FROM loans_shallow").collect()[0][0]
src_after   = spark.sql("SELECT count(*) FROM loans_constrained").collect()[0][0]

print(f"\nShallow clone after DELETE : {clone_after}  (closed rows removed)")
print(f"Source table unchanged     : {src_before} → {src_after}")


In [ ]:
for tbl in [
    "customers_delta", "loans", "loans_cdf", "txns_small",
    "txns_zorder", "txns_noorder", "txns_bloom", "txns_gen_col",
    "loans_constrained", "loans_shallow", "loans_deep",
]:
    spark.sql(f"DROP TABLE IF EXISTS {tbl}")

shutil.rmtree(SCRATCH, ignore_errors=True)
spark.stop()
